In [4]:
from google.colab import drive
drive.mount('/content/drive/')

import os
nli_dir = '/content/drive/MyDrive/nli'
os.makedirs(nli_dir, exist_ok=True)
os.chdir(nli_dir)

print(os.getcwd())

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
/content/drive/MyDrive/nli


In [ ]:
# Conditional pip install: exact requirements for bge (FlagEmbedding) if incomplete,
# exact requirements for Qwen3-Reranker if bge is done

import os
import pandas as pd
from nli_config import CODE_HYPOTHESES

expected_n_codes = len(CODE_HYPOTHESES)
bge_path = "reranker_scores_bge.csv"

bge_complete = False
if os.path.exists(bge_path):
    existing = pd.read_csv(bge_path)
    n_codes_done = existing["code_id"].nunique()
    bge_complete = n_codes_done >= expected_n_codes
    print(f"bge: {n_codes_done}/{expected_n_codes} codes scored")
else:
    print("bge: no output file found yet")

if not bge_complete:
    print("bge scoring not complete, installing FlagEmbedding's required transformers version...")
    !pip install -U "transformers==4.44.2" "FlagEmbedding>=1.3.2" "torch>=1.6.0" "accelerate>=0.20.1" "datasets>=2.19.0"
    print("\nRestart the runtime now (Runtime > Restart session), then re-run setup for reranker_bge.")
else:
    print("bge scoring complete, installing Qwen3-Reranker's required transformers version...")
    !pip uninstall -y transformers sentence-transformers
    !pip install -U "transformers>=4.51.0" "sentence-transformers>=4.1.0"
    print("\nRestart the runtime now (Runtime > Restart session), then re-run setup for reranker_qwen only.")

127 codes defined
exemplar=0 (expresses): 44
exemplar=1 (is an example of): 83
bge: 127/127 codes scored
bge scoring complete, installing Qwen3-Reranker's required transformers version...
Found existing installation: transformers 5.14.1
Uninstalling transformers-5.14.1:
  Successfully uninstalled transformers-5.14.1
Found existing installation: sentence-transformers 5.7.0
Uninstalling sentence-transformers-5.7.0:
  Successfully uninstalled sentence-transformers-5.7.0
  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)
  Using cached sentence_transformers-5.7.0-py3-none-any.whl.metadata (18 kB)
Using cached transformers-5.14.1-py3-none-any.whl (11.6 MB)
Using cached sentence_transformers-5.7.0-py3-none-any.whl (611 kB)

Restart the runtime now (Runtime > Restart session), then re-run setup for reranker_qwen only.


In [5]:
# setup bge-reranker-large (cross-encoder) or Qwen3-Reranker-0.6B, whichever is still needed

import os
import torch
import pandas as pd
from nli_config import CODE_HYPOTHESES

expected_n_codes = len(CODE_HYPOTHESES)
bge_path = "reranker_scores_bge.csv"

bge_complete = False
if os.path.exists(bge_path):
    existing = pd.read_csv(bge_path)
    n_codes_done = existing["code_id"].nunique()
    bge_complete = n_codes_done >= expected_n_codes

if not bge_complete:
    print("Loading bge-reranker-large...")
    from FlagEmbedding import FlagReranker
    reranker_bge = FlagReranker("BAAI/bge-reranker-large", use_fp16=True)
else:
    print("bge already complete, loading Qwen3-Reranker-0.6B...")
    from sentence_transformers import CrossEncoder
    reranker_qwen = CrossEncoder("Qwen/Qwen3-Reranker-0.6B", device="cuda")

judaism = pd.read_csv("ucberkeley-dlab_target_jewish.csv")

codebook = {code_id: definition for code_id, (definition, exemplar) in CODE_HYPOTHESES.items()}

print(f"Loaded {len(codebook)} codes")
print(f"Loaded {len(judaism)} comments")

bge already complete, loading Qwen3-Reranker-0.6B...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loaded 127 codes
Loaded 1874 comments


In [6]:
# Score full pilot corpus against all codes, both rerankers
# Only runs whichever model is currently loaded in the namespace (bge, qwen, or both)

import csv
import os
import torch
import pandas as pd
from tqdm.auto import tqdm

texts = judaism["text"].tolist()
comment_ids = judaism["comment_id"].tolist()
code_ids = list(CODE_HYPOTHESES.keys())
definitions = [CODE_HYPOTHESES[c][0] for c in code_ids]
expected_n = len(comment_ids)

RUNS = []
if "reranker_bge" in globals():
    RUNS.append(("bge", reranker_bge))
if "reranker_qwen" in globals():
    RUNS.append(("qwen", reranker_qwen))

if not RUNS:
    print("No reranker models loaded (reranker_bge / reranker_qwen not found). Run setup first.")

for model_label, model in RUNS:
    output_path = f"reranker_scores_{model_label}.csv"

    already_done = set()
    if os.path.exists(output_path):
        existing = pd.read_csv(output_path)
        counts = existing["code_id"].value_counts()
        already_done = set(counts[counts >= expected_n].index)
        incomplete = set(counts[counts < expected_n].index)
        if incomplete:
            print(f"[{output_path}] Found {len(incomplete)} incomplete codes, will be re-scored: {sorted(incomplete)}")
            existing = existing[~existing["code_id"].isin(incomplete)]
            existing.to_csv(output_path, index=False)
        print(f"[{output_path}] Resuming: {len(already_done)} codes fully scored, skipping those")

    remaining = [(c, d) for c, d in zip(code_ids, definitions) if c not in already_done]
    if not remaining:
        print(f"[{output_path}] Already complete, skipping run.")
        continue

    file_exists = os.path.exists(output_path)
    with open(output_path, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["comment_id", "code_id", "relevance_score"])

        pbar = tqdm(remaining, desc=f"Scoring codes ({model_label})", unit="code")
        for code_id, definition in pbar:
            pbar.set_postfix(code=code_id)

            if model_label == "bge":
                pairs = [[definition, text] for text in texts]
                scores = model.compute_score(pairs, batch_size=64, normalize=True)
            else:  # qwen
                pairs = [(definition, text) for text in texts]
                scores = model.predict(pairs, batch_size=64, activation_fn=torch.nn.Sigmoid())

            if len(scores) != expected_n:
                print(f"WARNING: {code_id} produced {len(scores)} scores, expected {expected_n}. Skipping write.")
                continue

            for cid, score in zip(comment_ids, scores):
                writer.writerow([cid, code_id, float(score)])
            f.flush()

    print(f"[{output_path}] All codes scored.")

[reranker_scores_qwen.csv] Resuming: 25 codes fully scored, skipping those


Scoring codes (qwen):   0%|          | 0/102 [00:00<?, ?code/s]

[reranker_scores_qwen.csv] All codes scored.
